# Análisis de Video — Pipeline de Tracking (Fase B)

Procesa el video de un partido y lo convierte en **tracking en coordenadas OPTA 0-100**
(el mismo sistema que SofaScore — todo se cruza directo).

**Flujo:** video → detección (YOLOv8) → tracking (ByteTrack) → equipos (clusters de
camiseta) → homografía (píxeles → cancha) → `tracking/positions.csv` → **validación
contra SofaScore**.

**Convención:** el video va en `data/temporadas/{año}/partidos/{match_key}/video/`
(cualquier formato: mp4, mkv, avi, mov).

In [1]:
import os, sys

# Recarga automática de módulos src/ al editarlos (sin reiniciar kernel)
%load_ext autoreload
%autoreload 2

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

import config
from src.video import (
    check_environment, environment_ready, process_video,
    load_tracking, track_summary, players_per_frame,
    compare_with_lineups, positions_plausibility,
)
from src.analysis.post_match import load_match_data

# ============================================================
# CONFIGURAR ACÁ
SEASON       = '2026'
MATCH_KEY    = '2026-04-19_vs_estudiantes_caseros_a'  # partido a procesar
TEST_SECONDS = 120    # primeros N segundos para validar; None = video completo
# ============================================================

VIDEO_DIR    = config.match_video_dir(MATCH_KEY, SEASON)
TRACKING_DIR = config.match_tracking_dir(MATCH_KEY, SEASON)

print(f'Partido  : {MATCH_KEY}')
print(f'Video en : {VIDEO_DIR}')
print(f'Tracking : {TRACKING_DIR}')

Partido  : 2026-04-19_vs_estudiantes_caseros_a
Video en : c:\Users\luart\OneDrive\Escritorio\Chaco For Ever Analisis\data\temporadas\2026\partidos\2026-04-19_vs_estudiantes_caseros_a\video
Tracking : c:\Users\luart\OneDrive\Escritorio\Chaco For Ever Analisis\data\temporadas\2026\partidos\2026-04-19_vs_estudiantes_caseros_a\tracking


## 1. Checklist del entorno

Antes de procesar hay que tener: dependencias instaladas, modelos descargados y el video del partido.

**Instalación (una sola vez, en una terminal con el venv activado):**
```bash
# PyTorch con CUDA para la GTX 1050 Ti
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
# El resto del stack CV
pip install -r requirements-video.txt
```

**Modelos:** los pesos `.pt` van en `models/`. Se obtienen del proyecto
[roboflow/sports](https://github.com/roboflow/sports) (ejemplo soccer): hay que
descargar los datasets de Roboflow Universe y entrenar, o conseguir pesos ya
entrenados de la comunidad. Nombres esperados (configurables en `config.VIDEO_MODELS`):
- `football-player-detection.pt`
- `football-pitch-detection.pt`
- `football-ball-detection.pt`

In [2]:
checks = check_environment(MATCH_KEY, SEASON)
for name, (ok, detail) in checks.items():
    print(f"{'✅' if ok else '❌'} {name}: {detail}")

LISTO = environment_ready(checks)
print(f"\n{'🟢 Entorno listo para procesar.' if LISTO else '🔴 Faltan requisitos (ver arriba).'}")

✅ PyTorch: instalado
✅ Ultralytics YOLO: instalado
✅ Supervision: instalado
✅ Roboflow sports: instalado
✅ OpenCV: instalado
✅ GPU: NVIDIA GeForce GTX 1050 Ti
❌ modelo player_detection: falta: c:\Users\luart\OneDrive\Escritorio\Chaco For Ever Analisis\models\football-player-detection.pt
❌ modelo pitch_detection: falta: c:\Users\luart\OneDrive\Escritorio\Chaco For Ever Analisis\models\football-pitch-detection.pt
❌ modelo ball_detection: falta: c:\Users\luart\OneDrive\Escritorio\Chaco For Ever Analisis\models\football-ball-detection.pt
✅ video del partido: c:\Users\luart\OneDrive\Escritorio\Chaco For Ever Analisis\data\temporadas\2026\partidos\2026-04-19_vs_estudiantes_caseros_a\video\2026-04-19_vs_estudiantes_caseros_a.mp4

🔴 Faltan requisitos (ver arriba).


## 2. Procesar el video

La primera vez usá `TEST_SECONDS = 120` (2 minutos) para validar que la extracción
funciona **antes** de procesar el partido entero. El partido completo a 5 fps de
muestreo puede tardar varias horas en la GTX 1050 Ti.

In [3]:
if LISTO:
    df_pos, df_ball, meta = process_video(
        MATCH_KEY, SEASON,
        max_seconds=TEST_SECONDS,   # None = video completo
        save=True,
    )
    print(f"\nPosiciones: {len(df_pos)} filas | Pelota: {len(df_ball)} filas")
else:
    # Si ya hay tracking guardado de una corrida anterior, lo cargamos
    tr = load_tracking(MATCH_KEY, SEASON)
    df_pos, df_ball, meta = tr['positions'], tr['ball'], tr['meta']
    if df_pos is not None:
        print(f"Tracking previo cargado: {len(df_pos)} filas ({meta.get('fecha','?')})")
    else:
        print('Sin entorno listo y sin tracking previo — completar el checklist.')

Sin entorno listo y sin tracking previo — completar el checklist.


## 3. Validación contra SofaScore

SofaScore es la verdad oficial: si el tracking se aleja mucho de los lineups
(jugadores, minutos) o la homografía pone puntos fuera de la cancha, la
extracción tiene problemas y NO hay que usarla para análisis.

In [4]:
if df_pos is not None and not df_pos.empty:
    # Plausibilidad espacial de la homografía
    pl = positions_plausibility(df_pos)
    print('Homografía:', '🟢 OK' if pl['ok'] else '🔴 REVISAR')
    print(f"  Dentro de la cancha: {pl['pct_dentro_cancha']}%")
    print(f"  Rango X: {pl['x_rango']} | Rango Y: {pl['y_rango']}")

    # Jugadores por frame (esperable ~10-11 por equipo con cámara de TV)
    print('\nJugadores detectados por frame:')
    print(players_per_frame(df_pos).to_string(index=False))

    # Cruce con lineups oficiales
    data = load_match_data(MATCH_KEY, SEASON)
    result = compare_with_lineups(df_pos, data['lineups'])
    if result:
        print('\nOficial SofaScore :', result['oficial_sofascore'])
        print('Tracking del video:', result['tracking_video'])
        if result['alertas']:
            for a in result['alertas']:
                print(f'⚠️  {a}')
        else:
            print('🟢 Sin alertas — tracking consistente con SofaScore.')

    # Resumen de tracks (top 25)
    print('\nTracks principales:')
    print(track_summary(df_pos).head(25).to_string(index=False))
else:
    print('Sin datos de tracking para validar.')

Sin datos de tracking para validar.


## 4. Vista rápida del tracking

Un instante del partido dibujado en la cancha: cada punto es un jugador en su
posición real según el video. Si esto se ve como una formación de fútbol
coherente, la homografía funciona.

In [5]:
if df_pos is not None and not df_pos.empty:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from mplsoccer import Pitch
    from src.visualization.season import BG_COLOR, TEXT_COLOR

    # Elegir un segundo del video para inspeccionar
    INSTANTE = float(df_pos['t_sec'].median())
    frame_df = df_pos[df_pos['t_sec'] == df_pos.loc[(df_pos['t_sec'] - INSTANTE).abs().idxmin(), 't_sec']]

    fig, ax = plt.subplots(figsize=(11, 7.5))
    fig.patch.set_facecolor(BG_COLOR)
    pitch = Pitch(pitch_type='opta', pitch_color='#1a3a1a', line_color='white', linewidth=1.2)
    pitch.draw(ax=ax)

    COLORS = {0: '#00d4aa', 1: '#cc3333', -1: '#FFD700'}
    for team_id, group in frame_df.groupby('team'):
        pitch.scatter(group['x'], group['y'], s=250, c=COLORS.get(team_id, 'white'),
                      edgecolor='white', linewidth=1.2, alpha=0.9, ax=ax,
                      label=f'Equipo {team_id}' if team_id >= 0 else 'Árbitro')

    if df_ball is not None and not df_ball.empty:
        b = df_ball.iloc[(df_ball['t_sec'] - INSTANTE).abs().idxmin()]
        pitch.scatter([b['x']], [b['y']], s=120, c='white', edgecolor='black',
                      linewidth=1.5, ax=ax, zorder=5, label='Pelota')

    ax.legend(facecolor=BG_COLOR, labelcolor=TEXT_COLOR, fontsize=9)
    ax.set_title(f'Tracking — segundo {INSTANTE:.0f} del video',
                 color=TEXT_COLOR, fontsize=13, fontweight='bold')
    plt.tight_layout()
    fig.savefig(os.path.join(TRACKING_DIR, 'vista_tracking.png'),
                dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    print('✅ vista_tracking.png guardada en tracking/')
    plt.close(fig)